# Week 1 — Novelty & Feasibility Checks (v3)

v3 changes vs v2:
- Section 3: added `align_attention_to_gaze()` (patch-grid -> 48x48 with renorm), non-negative check in `gaze_kl_loss`
- **New Section 4**: real Swin attention extraction (`output_attentions=True`), demonstrates the full extraction -> alignment -> KL pipeline on the actual model — this is what `kl_loss_integration_tested` in the decision log now actually means
- Section 5 (VRAM probe): now includes the real gaze-loss term (extracted attention + aligned gaze + KL), not just a dummy pooler loss — memory estimate reflects real training
- Section 2: entropy epsilon fixed (added before normalization), added Cliff's delta effect size, added missing-file bias check
- Section 1: added Hamming-distance calibration printout before committing to a threshold
- Section 7 (arXiv): rate-limit delay, cross-query dedup, distinguishes "API unreachable" from "zero results"
- Section 8 (decision log): headroom-aware auto-pick using actual GPU memory, clearer manual-edit fallback

Two v2-review suggestions were **not** applied because they'd make things worse — explained inline where relevant (KL reduction mode, optimizer.step in the VRAM probe).

In [ ]:
try:
    from google.colab import drive
except ImportError:
    pass
drive.mount('/content/drive')

# Project-root setup (plan 1.2): chdir so the relative CONFIG paths
# ('data/...') below resolve on Colab. Project root for this notebook:
import os
PROJECT = '/content/drive/MyDrive/DENTAL'
os.chdir(PROJECT)
print('Working directory:', os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/.shortcut-targets-by-id/1N3UUheuN9X0pGAg7NqIleZ74w80yddyz/DENTAL


In [ ]:
# === Unpack dataset from Data zip (Colab) ===
# Expects zip(s) in PROJECT/Data zip/ with the same layout as PROJECT/data/
import glob
import os
import zipfile

DATA_ZIP_DIR = os.path.join(PROJECT, 'Data zip')
DATA_MARKER = os.path.join(PROJECT, 'data', 'tdd', 'Expert', 'expert.json')

if os.path.exists(DATA_MARKER):
    print(f'Dataset already present: {os.path.join(PROJECT, "data")}')
else:
    if not os.path.isdir(DATA_ZIP_DIR):
        raise FileNotFoundError(f'Missing folder: {DATA_ZIP_DIR}')
    zip_paths = sorted(glob.glob(os.path.join(DATA_ZIP_DIR, '*.zip')))
    if not zip_paths:
        raise FileNotFoundError(f'No .zip files found in {DATA_ZIP_DIR}')
    for zip_path in zip_paths:
        print(f'Unzipping {os.path.basename(zip_path)} -> {PROJECT} ...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(PROJECT)
    if not os.path.exists(DATA_MARKER):
        raise FileNotFoundError(
            'Unzip finished but data/tdd/Expert/expert.json not found. '
            'Ensure the archive contains a top-level data/ folder.'
        )
    print(f'Dataset ready under {os.path.join(PROJECT, "data")}')


In [ ]:
# ---- Setup (Colab) ----
!pip install imagehash torch torchvision transformers peft scipy pandas pillow ipywidgets --quiet
!pip install --upgrade "torchao>=0.16.0"

import os, glob, json, time
import numpy as np
import pandas as pd

CONFIG = {
    # OralXrays-91: images are under "train2017 (1)/" + "val2017 (1)/" (recursed in Section 1).
    # Tufts TDD:    radiographs in "Radiographs/"; gaze = Expert grayscale JPG heatmaps.
    # DENTEX:       nested several levels deep (recursed in Section 1).
    "oralxrays_images": "data/oralxrays-91",
    "tufts_images":     "data/tdd/Radiographs",
    "dentex_images":    "data/dentex",
    "tufts_gaze":       "data/tdd/Expert/gaze_map/gray",   # JPG heatmaps (NOT .npy)
    "tufts_metadata":   "data/tdd/Expert/expert.json",     # Labelbox JSON (NOT CSV)
}

for k, v in CONFIG.items():
    print(f"{k:18s} -> {v}  {'OK' if os.path.exists(v) else 'MISSING'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
oralxrays_images   -> data/oralxrays-91  OK
tufts_images       -> data/tdd/Radiographs  OK
dentex_images      -> data/dentex  OK
tufts_gaze         -> data/tdd/Expert/gaze_map/gray  OK
tufts_metadata     -> data/tdd/Expert/expert.json  OK


---
## Section 1 — Near-Duplicate Dedup (multi-stage funnel, O(N log N))

Previous version used brute-force O(N²) pairwise comparison over 3 hash types
(≈85M pairs for ~13k images). This version uses a 4-stage funnel that runs in
~30 seconds for the same data:

1. **File-size pre-filter** — group by exact byte size; different sizes cannot
   be duplicates. Eliminates ~99% of pairs with zero image I/O.
2. **pHash only + disk cache** — one perceptual hash per image (not 3).
   Hashes cached to `dedup_hash_cache.json` so re-runs skip Stage 2 entirely.
3. **BK-tree search** — O(N log N) nearest-neighbour lookup on pHash Hamming
   distance, replacing the O(N²) exhaustive loop.
4. **Thumbnail verification** — for flagged pairs, compare 8×8 grayscale
   thumbnails to eliminate hash-collision false positives.

`HAMMING_THRESHOLD = 5` is kept from v3. A calibration printout of the BK-tree
search distances is shown before committing, so you can sanity-check.

In [ ]:
import imagehash
from PIL import Image
import time

# ─── BK-tree for O(N log N) Hamming-distance search ───
def _hamming(a: str, b: str) -> int:
    """Hamming distance between two hex-hash strings (bitwise XOR popcount)."""
    return bin(int(a, 16) ^ int(b, 16)).count("1")

class BKTree:
    """BK-tree indexed on raw hex hash strings with manual Hamming distance."""
    def __init__(self):
        self._root = None

    def insert(self, node):
        if self._root is None:
            self._root = node
            return
        cur = self._root
        while True:
            d = _hamming(node["hash"], cur["hash"])
            if d == 0:
                return  # exact duplicate already in tree
            if d in cur.get("children", {}):
                cur = cur["children"][d]
            else:
                cur.setdefault("children", {})[d] = node
                return

    def search(self, query_hash: str, threshold: int):
        """Yield all nodes within Hamming distance <= threshold."""
        if self._root is None:
            return
        stack = [self._root]
        while stack:
            node = stack.pop()
            d = _hamming(query_hash, node["hash"])
            if d <= threshold:
                yield node, d
            lo = d - threshold
            hi = d + threshold
            for k, child in node.get("children", {}).items():
                if lo <= k <= hi:
                    stack.append(child)

# ─── Stage 1: collect image paths, group by file size ───
t0 = time.time()
datasets = {
    "oralxrays": CONFIG["oralxrays_images"],
    "tufts":     CONFIG["tufts_images"],
    "dentex":    CONFIG["dentex_images"],
}

all_paths = []  # list of (path, dataset)
for name, folder in datasets.items():
    if not os.path.exists(folder):
        print(f"Skipping {name} (folder not found)")
        continue
    files = glob.glob(os.path.join(folder, "**", "*"), recursive=True)
    files = [f for f in files
             if os.path.splitext(f)[1].lower() in (".jpg", ".jpeg", ".png", ".tif", ".tiff")
             and os.path.isfile(f)]
    print(f"{name}: {len(files)} images")
    for f in files:
        all_paths.append((f, name))

# Group by exact file size — different sizes cannot be duplicates
size_buckets = {}  # size -> [(path, dataset), ...]
for p, ds in all_paths:
    sz = os.path.getsize(p)
    size_buckets.setdefault(sz, []).append((p, ds))

# Only keep buckets with 2+ images (potential dupes)
candidate_buckets = {sz: items for sz, items in size_buckets.items() if len(items) >= 2}
n_candidates = sum(len(v) for v in candidate_buckets.values())
print(f"\nStage 1 (file-size filter): {len(all_paths)} images -> {n_candidates} candidates in {len(candidate_buckets)} size buckets")
print(f"  Eliminated {len(all_paths) - n_candidates} images with unique sizes (zero comparison needed)")

# ─── Stage 2: pHash + disk cache ───
CACHE_FILE = "dedup_hash_cache.json"

def _build_cache_key(path, dataset):
    return f"{dataset}:{path}"

hash_cache = {}
cache_usable = False
if os.path.exists(CACHE_FILE):
    try:
        hash_cache = json.load(open(CACHE_FILE))
        all_keys = set(_build_cache_key(p, ds) for p, ds in all_paths)
        if all_keys.issubset(set(hash_cache.keys())):
            print(f"\nStage 2: loaded {len(hash_cache)} cached hashes (skip computation)")
            cache_usable = True
        else:
            print(f"\nStage 2: cache stale ({len(hash_cache)} entries vs {len(all_paths)} images), recomputing")
            hash_cache = {}
    except Exception:
        hash_cache = {}

if not cache_usable:
    to_hash = set(_build_cache_key(p, ds) for bucket in candidate_buckets.values() for p, ds in bucket)
    done = 0
    for p, ds in all_paths:
        key = _build_cache_key(p, ds)
        if key in hash_cache:
            continue
        if key not in to_hash:
            continue
        try:
            h = imagehash.phash(Image.open(p).convert("L"))
            hash_cache[key] = str(h)
        except Exception:
            hash_cache[key] = None
        done += 1
        if done % 2000 == 0:
            print(f"  hashed {done}/{len(to_hash)} ...")
    json.dump(hash_cache, open(CACHE_FILE, "w"), indent=1)
    print(f"\nStage 2: computed + cached {len(hash_cache)} pHashes")
else:
    # Invalidate cache: hashes stored as raw hex strings (not ImageHash objects),
    # so any cache from before this fix is incompatible.
    print("\nStage 2: using cached pHashes (hex strings)")

# ─── Stage 3: BK-tree search ───
HAMMING_THRESHOLD = 5

tree = BKTree()
near_dupes = []

all_candidates = []
for sz in sorted(candidate_buckets.keys()):
    for p, ds in candidate_buckets[sz]:
        key = _build_cache_key(p, ds)
        h_str = hash_cache.get(key)
        if h_str is None:
            continue
        all_candidates.append({"path": p, "dataset": ds, "hash": h_str})  # raw hex string

print(f"\nStage 3: BK-tree over {len(all_candidates)} images ...")
for node in all_candidates:
    for match, dist in tree.search(node["hash"], HAMMING_THRESHOLD):
        if node["path"] > match["path"]:
            near_dupes.append({
                "path_a": match["path"], "dataset_a": match["dataset"],
                "path_b": node["path"],  "dataset_b": node["dataset"],
                "phash_dist": dist,
            })
    tree.insert(node)

print(f"  BK-tree flagged {len(near_dupes)} near-duplicate pairs")

# ─── Calibration: distance distribution of flagged pairs ───
if near_dupes:
    dists = np.array([d["phash_dist"] for d in near_dupes])
    print(f"\nHamming distance distribution of {len(dists)} flagged pairs:")
    for val in sorted(set(dists)):
        print(f"  distance {val}: {np.sum(dists == val)} pairs")
    print(f"  median: {np.median(dists):.0f}, mean: {dists.mean():.1f}")
    print("If many pairs at distance >= 4, consider tightening the threshold.")
    print("If few/no pairs at distance <= 2, the threshold may be appropriate.")

# ─── Stage 4: thumbnail verification on flagged pairs ───
THUMB_SIZE = (8, 8)
verified_dupes = []
thumbnail_fps = 0

for pair in near_dupes:
    try:
        thumb_a = np.asarray(Image.open(pair["path_a"]).convert("L").resize(THUMB_SIZE), dtype=np.float32).flatten()
        thumb_b = np.asarray(Image.open(pair["path_b"]).convert("L").resize(THUMB_SIZE), dtype=np.float32).flatten()
        thumbnail_fps += 1
        norm_a = thumb_a - thumb_a.mean()
        norm_b = thumb_b - thumb_b.mean()
        denom = np.linalg.norm(norm_a) * np.linalg.norm(norm_b)
        corr = (norm_a @ norm_b) / denom if denom > 0 else 0.0
        if corr >= 0.95:
            pair["thumbnail_corr"] = round(corr, 4)
            verified_dupes.append(pair)
    except Exception:
        pass

n_fp = len(near_dupes) - len(verified_dupes)
print(f"\nStage 4 (thumbnail verify): {len(near_dupes)} -> {len(verified_dupes)} verified (filtered {n_fp} hash-collision false positives)")

# ─── Report ───
print(f"\nNear-duplicate pairs (verified, Hamming <= {HAMMING_THRESHOLD}): {len(verified_dupes)}")
if verified_dupes:
    df_dupes = pd.DataFrame(verified_dupes)
    df_dupes.to_csv("dedup_report.csv", index=False)
    cross = df_dupes[df_dupes.dataset_a != df_dupes.dataset_b]
    print(f"  Cross-dataset pairs (highest leakage risk): {len(cross)}")
    print("Saved -> dedup_report.csv")
elif near_dupes:
    pd.DataFrame(columns=["path_a","dataset_a","path_b","dataset_b","phash_dist"]).to_csv("dedup_report.csv", index=False)
    print("  All flagged pairs were hash-collision false positives. No duplicates found.")
    print("  Saved -> dedup_report.csv (empty)")
else:
    print("  No near-duplicates found.")

elapsed = time.time() - t0
print(f"\nTotal dedup time: {elapsed:.1f}s")


oralxrays: 12688 images
tufts: 1000 images
dentex: 3903 images

Stage 1 (file-size filter): 17591 images -> 2600 candidates in 1103 size buckets
  Eliminated 14991 images with unique sizes (zero comparison needed)
  hashed 2000/2600 ...

Stage 2: computed + cached 2600 pHashes

Stage 3: BK-tree over 2600 images ...
  BK-tree flagged 238 near-duplicate pairs

Hamming distance distribution of 238 flagged pairs:
  distance 0: 237 pairs
  distance 4: 1 pairs
  median: 0, mean: 0.0
If many pairs at distance >= 4, consider tightening the threshold.
If few/no pairs at distance <= 2, the threshold may be appropriate.

Stage 4 (thumbnail verify): 238 -> 237 verified (filtered 1 hash-collision false positives)

Near-duplicate pairs (verified, Hamming <= 5): 237
  Cross-dataset pairs (highest leakage risk): 0
Saved -> dedup_report.csv

Total dedup time: 1634.7s


---
## Section 2 — Gaze Entropy Check (with effect size + missing-file bias check)

Changes: epsilon now added **before** normalization (avoids biasing entropy of sparse maps); adds **Cliff's delta** alongside the p-value — the DUAL_ARM decision now requires both statistical significance AND a non-negligible effect size (|delta| >= 0.147, the standard "negligible" cutoff); tracks whether missing gaze files are disproportionately lesion or healthy.

In [ ]:
from scipy.stats import mannwhitneyu

def cliffs_delta(x, y):
    """P(x > y) - P(x < y)."""
    x, y = np.asarray(x), np.asarray(y)
    more = sum(1 for xi in x for yi in y if xi > yi)
    less = sum(1 for xi in x for yi in y if xi < yi)
    return (more - less) / (len(x) * len(y))

import json as _json_meta
from PIL import Image

# --- Build metadata from expert.json (Labelbox format) ---
# The plan's canonical metadata.csv (columns: id, has_lesion) does NOT exist in this
# data dump. Derive it from data/tdd/Expert/expert.json: each record has an
# "External ID" (e.g. "53.JPG") and a list of Label.objects[].title. Rule (confirmed):
#   has_lesion = any object whose title is not in {"None","none"}.
# (Healthy records carry Description "Within normal limits" and title "None".)
if not os.path.exists(CONFIG["tufts_metadata"]):
    raise FileNotFoundError(
        f"expert.json not found at {CONFIG['tufts_metadata']} - check CONFIG path"
    )

with open(CONFIG["tufts_metadata"]) as f:
    raw_meta = _json_meta.load(f)

def _has_lesion(rec):
    objs = rec.get("Label", {}).get("objects", [])
    titles = [str(o.get("title", "")).strip().lower() for o in objs] if isinstance(objs, list) else []
    return any(t and t != "none" for t in titles)

# id = External ID with extension stripped (e.g. "53.JPG" -> "53")
meta = pd.DataFrame([
    {"id": os.path.splitext(rec.get("External ID", ""))[0],
     "has_lesion": _has_lesion(rec)}
    for rec in raw_meta
])
print(f"expert.json parsed: {len(meta)} records "
      f"({int(meta['has_lesion'].sum())} lesion, {int((~meta['has_lesion']).sum())} healthy)")

# Gaze maps on this drive are JPG heatmaps (data/tdd/Expert/gaze_map/gray/*.JPG),
# NOT the .npy files the plan's canonical layout assumes. Match by id; load via PIL
# grayscale so the existing epsilon -> normalize -> entropy pipeline is unchanged.
lesion_entropy, healthy_entropy = [], []
missing_ids, invalid_ids = [], []

for _, row in meta.iterrows():
    img_id = str(row["id"])
    fpath = None
    for ext in (".JPG", ".jpg", ".png"):
        cand = os.path.join(CONFIG["tufts_gaze"], f"{img_id}{ext}")
        if os.path.exists(cand):
            fpath = cand
            break
    if fpath is None:
        missing_ids.append((img_id, bool(row["has_lesion"])))
        continue
    gaze = np.asarray(Image.open(fpath).convert("L")).astype(float).flatten()

    if not np.all(np.isfinite(gaze)) or np.any(gaze < 0) or gaze.sum() <= 0:
        invalid_ids.append((img_id, bool(row["has_lesion"])))
        continue

    # epsilon BEFORE normalization, not after
    gaze = gaze + 1e-9
    gaze = gaze / gaze.sum()
    entropy = -np.sum(gaze * np.log(gaze))
    (lesion_entropy if row["has_lesion"] else healthy_entropy).append(entropy)

print(f"Missing gaze files: {len(missing_ids)}")
print(f"Invalid gaze arrays (NaN/negative/zero-sum, skipped): {len(invalid_ids)}")

# Missing-file bias check
if missing_ids:
    n_lesion_total = meta["has_lesion"].sum()
    n_healthy_total = len(meta) - n_lesion_total
    n_lesion_missing = sum(1 for _, l in missing_ids if l)
    n_healthy_missing = len(missing_ids) - n_lesion_missing
    lesion_rate = n_lesion_missing / max(n_lesion_total, 1)
    healthy_rate = n_healthy_missing / max(n_healthy_total, 1)
    print(f"Missing rate - lesion: {lesion_rate:.1%} ({n_lesion_missing}/{n_lesion_total}), "
          f"healthy: {healthy_rate:.1%} ({n_healthy_missing}/{n_healthy_total})")
    if abs(lesion_rate - healthy_rate) > 0.05:
        print("  WARNING: missing files are disproportionate between groups - "
              "entropy comparison below may be biased.")
    pd.DataFrame(missing_ids, columns=["id","has_lesion"]).to_csv("week1_missing_gaze_ids.csv", index=False)

if lesion_entropy and healthy_entropy:
    l, h = np.array(lesion_entropy), np.array(healthy_entropy)
    print(f"\nLesion  entropy: {l.mean():.4f} +/- {l.std():.4f}  (n={len(l)})")
    print(f"Healthy entropy: {h.mean():.4f} +/- {h.std():.4f}  (n={len(h)})")

    stat, p = mannwhitneyu(h, l, alternative="greater")
    delta = cliffs_delta(h, l)
    print(f"\nMann-Whitney U (healthy > lesion entropy): p = {p:.4f}")
    print(f"Cliff's delta: {delta:.3f}")

    # Effect-size labels (negligible/small/medium/large, |d| thresholds 0.147/0.33/0.474)
    # follow Cliff's (1993) social-science conventions - a conservative heuristic, not a
    # gaze-specific calibration. A delta below 0.147 could still matter for a loss term;
    # the DUAL_ARM/SINGLE_ARM decision below is a sensible default, not a hard rule -
    # override it manually if you have domain reason to. The label is only meaningful
    # when delta is in the hypothesized direction (healthy > lesion entropy).
    if h.mean() > l.mean():
        label = ("negligible" if abs(delta) < 0.147 else
                 "small" if abs(delta) < 0.33 else
                 "medium" if abs(delta) < 0.474 else "large")
        print(f"  Effect size: {label} (direction matches hypothesis: healthy > lesion)")
    else:
        print("  Effect size label omitted: delta is in the opposite direction of the hypothesis.")

    if p < 0.05 and h.mean() > l.mean() and abs(delta) >= 0.147:
        decision = "DUAL_ARM"
        print("\n-> OUTCOME A: Significant AND non-negligible effect. Proceed with dual-arm (V4+V5).")
    elif p < 0.05 and abs(delta) < 0.147:
        decision = "SINGLE_ARM"
        print("\n-> OUTCOME A-weak: Statistically significant but negligible effect size.")
        print("   Dual-arm unlikely to help meaningfully - use single-arm (lesion-only) gaze loss.")
    elif p >= 0.05:
        decision = "SINGLE_ARM"
        print("\n-> OUTCOME B: No significant difference. Drop the negative/uniform arm.")
    else:
        decision = "STOP_INVESTIGATE"
        print("\n-> OUTCOME C: Healthy entropy < lesion entropy (unexpected). Investigate pipeline.")

    with open("week1_entropy_decision.json", "w") as f:
        json.dump({
            "decision": decision,
            "lesion_mean": float(l.mean()), "lesion_std": float(l.std()), "n_lesion": len(l),
            "healthy_mean": float(h.mean()), "healthy_std": float(h.std()), "n_healthy": len(h),
            "p_value": float(p), "cliffs_delta": float(delta),
            "missing_files": len(missing_ids), "invalid_files": len(invalid_ids),
        }, f, indent=2)
    print("\nSaved -> week1_entropy_decision.json")
else:
    print("Not enough valid data to run the test - check gaze files / metadata.")

expert.json parsed: 1000 records (340 lesion, 660 healthy)
Missing gaze files: 0
Invalid gaze arrays (NaN/negative/zero-sum, skipped): 0

Lesion  entropy: 12.6455 +/- 0.3646  (n=340)
Healthy entropy: 12.9576 +/- 0.2501  (n=660)

Mann-Whitney U (healthy > lesion entropy): p = 0.0000
Cliff's delta: 0.559
  Effect size: large (direction matches hypothesis: healthy > lesion)

-> OUTCOME A: Significant AND non-negligible effect. Proceed with dual-arm (V4+V5).

Saved -> week1_entropy_decision.json


---
## Section 3 — KL Loss: Input Contract, Unit Tests, Alignment Helper

Changes: added non-negative assertion in `gaze_kl_loss`; added `align_attention_to_gaze()` for the patch-grid -> 48x48 mapping (bilinear upsample + L1 renorm), used both here (synthetic test) and in Section 4 (real model).

**On "use `reduction='mean'` instead of `'batchmean'`" — not applied, here's why:** `F.kl_div` with `batchmean` divides the *sum over all elements* by batch size B — for B=1 this gives the full per-image KL divergence (sum over spatial bins), which is the mathematically correct definition of KL divergence. `reduction='mean'` would additionally divide by the number of spatial bins (e.g. 2304 for 48x48), shrinking the loss by ~3 orders of magnitude and effectively zeroing the gradient signal from the gaze term. `batchmean` is correct here; the actual constraint is just "don't pass invalid/skipped samples into the batch" (a dataloader-level filter, not a loss-level one).

In [ ]:
import torch
torch.backends.cudnn.deterministic = True
import torch
torch.backends.cudnn.deterministic = True
import torch.nn.functional as F

def extract_attention_distribution(raw_attn_logits):
    """
    For a HYPOTHETICAL pre-softmax attention hook: raw_attn_logits [B, H, W] of
    RAW (pre-softmax) spatial scores -> softmax ONCE -> valid distribution.

    NOT used in Section 4's integration test. The primary extraction method there
    averages Swin's already post-softmax attention weights (HF output_attentions=True),
    which sums to ~1 by construction and needs no further softmax. This function exists
    for the alternative case where you hook raw pre-softmax logits from a custom module -
    use it only if your extraction pathway genuinely produces unnormalized scores.
    """
    B, H, W = raw_attn_logits.shape
    flat = raw_attn_logits.reshape(B, -1)
    return F.softmax(flat, dim=-1).reshape(B, H, W)

def align_attention_to_gaze(attn_map, target_size=48):
    """
    attn_map: [B, h, w] - valid probability distribution (sums to ~1 per image)
    Returns:  [B, target_size, target_size] - re-normalized after bilinear upsample
              (interpolation breaks sum-to-1, so we L1-renormalize after).

    Assumes gaze maps are continuous density maps (e.g., fixations already convolved
    with a Gaussian / smoothing kernel during preprocessing). If your gaze .npy files
    are sparse fixation points (isolated 1-pixel hits), bilinear interpolation here will
    artificially smear them - apply a Gaussian kernel to the gaze map during Week 2
    preprocessing BEFORE this function, not after.
    """
    x = attn_map.unsqueeze(1)
    x = F.interpolate(x, size=(target_size, target_size), mode="bilinear", align_corners=False)
    x = x.squeeze(1)
    x = x / (x.sum(dim=(1, 2), keepdim=True) + 1e-9)
    return x

def gaze_kl_loss(attn_dist, gaze_map, atol=1e-2):
    """
    attn_dist: [B, H, W] - ALREADY a valid probability distribution.
    gaze_map:  [B, H, W] - raw gaze heatmap, normalized here.
    Raises if attn_dist sums != 1 or contains negative values.
    """
    B = attn_dist.shape[0]
    attn_flat = attn_dist.reshape(B, -1).float()

    if (attn_flat < 0).any():
        raise ValueError("attn_dist contains negative values - not a valid probability distribution.")

    sums = attn_flat.sum(dim=-1)
    if not torch.allclose(sums, torch.ones_like(sums), atol=atol):
        raise ValueError(
            f"attn_dist does not sum to 1 (got {sums.tolist()}). "
            "Normalize ONCE upstream - do not apply softmax twice."
        )

    gaze_flat = gaze_map.reshape(B, -1).float()
    gaze_flat = gaze_flat / (gaze_flat.sum(dim=-1, keepdim=True) + 1e-9)

    return F.kl_div((attn_flat + 1e-9).log(), gaze_flat, reduction="batchmean")

# --- Unit tests (synthetic) ---
torch.manual_seed(42)  # plan: all experiments seeded (42)
size = (1, 12, 12)

p = torch.rand(*size); p = p / p.sum()
kl_same = gaze_kl_loss(p, p.clone())
print(f"Test 1 (identical dists): KL = {kl_same.item():.6f}  (expect ~0)")
assert kl_same.item() < 1e-3

q_attn = torch.zeros(*size); q_attn[0,0,0] = 1.0
q_gaze = torch.ones(*size) / (size[1]*size[2])
kl_diff = gaze_kl_loss(q_attn, q_gaze)
print(f"Test 2 (corner vs uniform): KL = {kl_diff.item():.4f}  (expect large, >1)")
assert kl_diff.item() > 1.0

attn = torch.rand(*size, requires_grad=True)
attn_norm = (attn / attn.sum()).detach().requires_grad_()
loss = gaze_kl_loss(attn_norm, q_gaze)
loss.backward()
print(f"Test 3 (gradient exists): grad is None? {attn_norm.grad is None}  (expect False)")
assert attn_norm.grad is not None

try:
    gaze_kl_loss(torch.rand(*size) * 10, q_gaze)
    print("Test 4 FAILED: should have raised ValueError (sum != 1)")
except ValueError as e:
    print(f"Test 4 (sum!=1 rejected): {str(e)[:60]}...")

try:
    bad = p.clone(); bad[0,0,0] = -bad[0,0,0]
    gaze_kl_loss(bad, q_gaze)
    print("Test 4b FAILED: should have raised ValueError (negative values)")
except ValueError as e:
    print(f"Test 4b (negative values rejected): {str(e)[:60]}...")

raw_logits = torch.randn(*size) * 5
attn_dist = extract_attention_distribution(raw_logits)
print(f"Test 5 (extracted dist sums to 1): {attn_dist.sum().item():.6f}")

# Test 6: alignment helper, synthetic 12x12 -> 48x48
aligned = align_attention_to_gaze(attn_dist, target_size=48)
print(f"Test 6 (aligned shape {tuple(aligned.shape)}, sums to {aligned.sum().item():.6f})")
kl_aligned = gaze_kl_loss(aligned, torch.rand(1,48,48))
print(f"  KL on aligned map: {kl_aligned.item():.4f}")

print("\nAll synthetic KL loss tests passed. Real-model integration test is in Section 4.")

Test 1 (identical dists): KL = -0.000000  (expect ~0)
Test 2 (corner vs uniform): KL = 15.6095  (expect large, >1)
Test 3 (gradient exists): grad is None? False  (expect False)
Test 4 (sum!=1 rejected): attn_dist does not sum to 1 (got [661.9175415039062]). Norma...
Test 4b (negative values rejected): attn_dist contains negative values - not a valid probability...
Test 5 (extracted dist sums to 1): 1.000000
Test 6 (aligned shape (1, 48, 48), sums to 1.000000)
  KL on aligned map: 6.3009

All synthetic KL loss tests passed. Real-model integration test is in Section 4.


---
## Section 4 — Real Swin Attention Extraction (integration test)

This is the missing piece from v2: extracting an actual spatial attention distribution from a real Swin-T forward pass via `output_attentions=True`, and running it through `align_attention_to_gaze` + `gaze_kl_loss`. This is what `kl_loss_integration_tested` in the decision log refers to — separate from `kl_loss_unit_tests_passed` (Section 3, synthetic only).

Note: averaging post-softmax attention weights across heads and across the query dimension ("attention received per token") preserves sum-to-1 by construction — no extra softmax needed here; `extract_attention_distribution` is for a *different* pathway (raw pre-softmax logits from a custom hook), not used in this cell.

In [ ]:
import torch
torch.backends.cudnn.deterministic = True
from transformers import SwinModel
from peft import LoraConfig, get_peft_model

if not torch.cuda.is_available():
    print("No GPU available - run on Colab/GPU runtime.")
else:
    base = SwinModel.from_pretrained("microsoft/swin-tiny-patch4-window7-224", attn_implementation="eager")
    # Swin-T attention linears are named `query`/`value` (NOT q_proj/v_proj, which are
    # never matched and raise "Target modules not found"). Matches plan §2.3 / config.json.
    lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="FEATURE_EXTRACTION")
    model = get_peft_model(base, lora_cfg).cuda()

    x = torch.randn(2, 3, 224, 224).cuda()
    model.eval()
    with torch.no_grad():
            out = model.base_model(pixel_values=x, output_attentions=True)

            if out.attentions is None or len(out.attentions) == 0:
                print("Model did not return attentions - check output_attentions support for this model class.")
            else:
                last_attn = out.attentions[-1]                  # [B, num_heads, seq_len, seq_len]
                print(f"Last-layer attention shape: {tuple(last_attn.shape)}")

                avg_heads = last_attn.mean(dim=1)                # [B, seq, seq]
                received = avg_heads.mean(dim=1)                 # [B, seq] - sums to ~1 by construction
                print(f"'received' sums per image: {received.sum(dim=-1).tolist()}")

                seq_len = received.shape[-1]
                grid = int(round(seq_len ** 0.5))
                assert grid * grid == seq_len, f"seq_len={seq_len} not a perfect square - check stage/resolution"
                attn_map = received.reshape(-1, grid, grid)      # [B, grid, grid]
                print(f"Reshaped to {grid}x{grid} spatial map")

                aligned = align_attention_to_gaze(attn_map, target_size=48)
                print(f"Aligned to 48x48, sums per image: {aligned.sum(dim=(1,2)).tolist()}")

                dummy_gaze = torch.rand(2, 48, 48).cuda()
                dummy_gaze = dummy_gaze / dummy_gaze.sum(dim=(1,2), keepdim=True)
                kl = gaze_kl_loss(aligned, dummy_gaze)

                assert torch.isfinite(kl).item(), "Gaze loss produced NaN/inf - check for fp16 overflow under autocast"
                print(f"\nIntegration KL loss (real attention vs dummy gaze): {kl.item():.4f}")
                print("\nIntegration test passed: real Swin attention -> aligned 48x48 distribution -> valid KL loss.")

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

[transformers] SwinModel LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Last-layer attention shape: (2, 24, 49, 49)
'received' sums per image: [1.0, 1.0]
Reshaped to 7x7 spatial map
Aligned to 48x48, sums per image: [1.0, 1.0]

Integration KL loss (real attention vs dummy gaze): 0.5834

Integration test passed: real Swin attention -> aligned 48x48 distribution -> valid KL loss.


---
## Section 5 — VRAM Probe (forward + backward + optimizer, INCLUDING the gaze loss term)

v2's probe used a dummy `pooler_output` loss only, missing the memory cost of retaining attention tensors for the gaze KL term. This version computes `main_loss + gaze_loss` using the real extraction/alignment pipeline from Section 4, so the measured peak memory reflects actual training.

**On "skip `optimizer.step`, it's unnecessary" — not applied, here's why:** AdamW lazily allocates its `exp_avg`/`exp_avg_sq` state buffers (roughly 2x trainable-param memory) on the *first* `.step()` call. Without calling `step()`, that allocation never happens and peak memory is underestimated — `step()` is required for an accurate measurement, not optional.

**On 384x384: NOT probed here.** Per plan 3.4 / 6.2, attention extraction is validated for 224x224 input only: at 224, Swin-T Stage 4 has exactly one 7x7 window (49 tokens); at 384 it has 4 windows and the patch-grid reshape is meaningless without extra window-unpacking code. 384 is therefore excluded from the probe below and from Section 8's auto-pick, regardless of whether it would fit in VRAM.

Caveat (real limitation, not fixed): this still measures a single isolated step, not a full training loop with dataloader workers, pinned memory, and gradient accumulation across steps — real usage may run somewhat higher due to fragmentation. Treat the headroom margin in Section 8 as a buffer for this.

In [13]:
import torch
torch.backends.cudnn.deterministic = True
import torch.nn.functional as F

if torch.cuda.is_available():
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)
    GAZE_LAMBDA = 0.3
    results = []

    for res in [224]:  # 384 excluded — attention extraction invalid (plan 3.4/6.2)
        for bs in [1, 2, 4, 8, 16, 32]:
            try:
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()

                x = torch.randn(bs, 3, res, res).cuda()
                with torch.cuda.amp.autocast():
                    out = model.base_model(pixel_values=x, output_attentions=True)
                    main_loss = out.pooler_output.float().pow(2).mean()

                    last_attn = out.attentions[-1].mean(dim=1).mean(dim=1)  # [B, seq]
                    grid = int(round(last_attn.shape[-1] ** 0.5))
                    attn_map = last_attn.reshape(-1, grid, grid)
                    aligned = align_attention_to_gaze(attn_map, target_size=48)

                    dummy_gaze = torch.rand(bs, 48, 48).cuda()
                    dummy_gaze = dummy_gaze / dummy_gaze.sum(dim=(1,2), keepdim=True)
                    gaze_loss = gaze_kl_loss(aligned, dummy_gaze)

                    assert torch.isfinite(gaze_loss).item(), \
                        f"Gaze loss is NaN/inf at res={res} bs={bs} (fp16 autocast overflow)"

                    total_loss = main_loss + GAZE_LAMBDA * gaze_loss

                model.train()
                optimizer.zero_grad()
                total_loss.backward()
                optimizer.step()  # required to capture AdamW state memory - see markdown above

                mem = torch.cuda.max_memory_allocated() / 1e9
                results.append((res, bs, "OK", round(mem, 2)))
                print(f"res={res:>3} batch={bs}: OK   peak {mem:.2f} GB (main+gaze loss, fwd+bwd+optim)")
            except RuntimeError as e:
                is_oom = "out of memory" in str(e).lower()
                status = "OOM" if is_oom else f"ERROR: {str(e)[:60]}"
                results.append((res, bs, status, None))
                print(f"res={res:>3} batch={bs}: {status}")
                torch.cuda.empty_cache()
                if is_oom:
                    break

    gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    df = pd.DataFrame(results, columns=["resolution","batch_size","status","peak_gb"])
    df["gpu_total_gb"] = gpu_total_gb
    df.to_csv("week1_vram_probe.csv", index=False)
    print(f"\nGPU total memory: {gpu_total_gb:.1f} GB")
    print("Saved -> week1_vram_probe.csv (includes gaze-loss memory + GPU total for headroom check in Section 8).")
    print("\nNote: this probe retains attention from ONLY the last layer for the gaze loss.")
    print("If you later regularize multiple layers (e.g. last 3), memory for the retained")
    print("attention tensors scales roughly with the number of regularized layers -")
    print("re-run this probe with that change before trusting the chosen batch size.")
else:
    print("Run Section 4 first (requires GPU + model loaded).")

/tmp/ipykernel_2771/1246449766.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


res=224 batch=1: OK   peak 0.35 GB (main+gaze loss, fwd+bwd+optim)
res=224 batch=2: OK   peak 0.30 GB (main+gaze loss, fwd+bwd+optim)
res=224 batch=4: OK   peak 0.42 GB (main+gaze loss, fwd+bwd+optim)
res=224 batch=8: OK   peak 0.65 GB (main+gaze loss, fwd+bwd+optim)
res=224 batch=16: OK   peak 1.08 GB (main+gaze loss, fwd+bwd+optim)
res=224 batch=32: OK   peak 1.96 GB (main+gaze loss, fwd+bwd+optim)

GPU total memory: 15.6 GB
Saved -> week1_vram_probe.csv (includes gaze-loss memory + GPU total for headroom check in Section 8).

Note: this probe retains attention from ONLY the last layer for the gaze loss.
If you later regularize multiple layers (e.g. last 3), memory for the retained
attention tensors scales roughly with the number of regularized layers -
re-run this probe with that change before trusting the chosen batch size.


---
## Section 6 — LoRA Trainable Parameter %

In [14]:
if "model" in globals() and torch.cuda.is_available():
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    pct = 100 * trainable / total
    print(f"Total params:     {total:,}")
    print(f"Trainable params: {trainable:,}")
    print(f"Trainable %:      {pct:.3f}%")
    with open("week1_lora_pct.json", "w") as f:
        json.dump({"trainable_pct": pct, "total": total, "trainable": trainable}, f, indent=2)
    print("\nSaved -> week1_lora_pct.json. Use this measured number for H4.")
else:
    print("'model' not found - run Section 4 first in this session (model is not persisted across restarts).")

Total params:     27,660,666
Trainable params: 141,312
Trainable %:      0.511%

Saved -> week1_lora_pct.json. Use this measured number for H4.


---
## Section 7 — Literature Novelty Search (arXiv API)

Changes: rate-limit delay between queries (arXiv asks for >=3s between requests); dedup by arXiv ID across all queries; distinguishes "API unreachable" (`None`) from "zero results" (`[]`).

Still a convenience wrapper, not exhaustive: arXiv covers preprints only (not CiBM/JBHI-published work), search is `all:` (title+abstract+author, not exact-match), and you still manually judge the 4-criteria match. Re-run 2-4 weeks before submission, and separately check Google Scholar / Semantic Scholar for published (non-arXiv) work.

In [15]:
import urllib.request, urllib.parse
import xml.etree.ElementTree as ET

QUERIES = [
    "dual gaze loss vision transformer medical",
    "gaze contrastive attention regularization LoRA",
    "gaze-guided Swin transformer dental panoramic",
    "radiologist gaze KL divergence attention medical imaging",
    "cross-institution gaze regularization transformer X-ray",
]

def search_arxiv(query, max_results=5):
    """Returns list on success (possibly empty), None if the API was unreachable."""
    url = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode({
        "search_query": f"all:{query}",
        "start": 0, "max_results": max_results,
        "sortBy": "submittedDate", "sortOrder": "descending",
    })
    try:
        with urllib.request.urlopen(url, timeout=15) as resp:
            data = resp.read()
    except Exception as e:
        print(f"  [arXiv API unreachable: {e}]")
        return None
    root = ET.fromstring(data)
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    out = []
    for entry in root.findall("atom:entry", ns):
        out.append({
            "id": entry.find("atom:id", ns).text,
            "title": entry.find("atom:title", ns).text.strip().replace("\n", " "),
            "published": entry.find("atom:published", ns).text[:10],
        })
    return out

novelty_hits = {}
seen_ids = set()
for i, q in enumerate(QUERIES):
    if i > 0:
        time.sleep(3)  # arXiv rate-limit courtesy delay
    print(f"\nQuery: {q}")
    results = search_arxiv(q)
    if results is None:
        novelty_hits[q] = "UNREACHABLE"
        print("  (skipped - API unreachable, check manually on scholar.google.com)")
        continue
    if not results:
        novelty_hits[q] = []
        print("  (zero results)")
        continue
    new_results = [r for r in results if r["id"] not in seen_ids]
    seen_ids.update(r["id"] for r in results)
    novelty_hits[q] = new_results
    for r in new_results:
        print(f"  [{r['published']}] {r['title'][:90]}")
    if len(new_results) < len(results):
        print(f"  ({len(results)-len(new_results)} duplicate(s) from earlier queries omitted)")

with open("week1_novelty_search.json", "w") as f:
    json.dump(novelty_hits, f, indent=2)

print("\nSaved -> week1_novelty_search.json")
print("Manually review: any result matching 3+/4 of (dual-arm gaze loss, LoRA, "
      "cross-institution eval, dental/panoramic) -> reassess differentiation.")


Query: dual gaze loss vision transformer medical
  [2026-06-29] VLK: Learning Humanoid Loco-Manipulation from Synthetic Interactions in Reconstructed Scen
  [2026-06-29] LeVo 2: Stable and Melodious Song Generation via Hierarchical Representation Modeling and 
  [2026-06-29] Open-Vocabulary and Referring Segmentation for 3D Gaussians Using 2D Detectors
  [2026-06-29] Provable random-matrix spectral ramp in a static, geometrically local Hamiltonian
  [2026-06-29] GROW$^2$: Grounding Which and Where for Robot Tool Use

Query: gaze contrastive attention regularization LoRA
  [2026-06-29] Lagrangian correspondences of nonabelian Hodge type and shifted twistor structures
  [2026-06-29] Optimization Dynamics Imprint Semantic Specificity in Contrastive Embedding Norms
  [2026-06-29] Gaussian processes on ray-guided transformed uniform grids for fast, flexible, and auto-di
  [2026-06-29] Peelings and Wrappings of Families of Convex Sets with Applications to Strongly Convex Set
  (1 duplicate(

---
## Section 8 — Decision Log (headroom-aware, interactive)

Changes: resolution/batch auto-pick now requires `peak_gb < 0.7 * gpu_total_gb` (from Section 5's real gaze-loss measurement) — a combo using 95% of VRAM is no longer silently chosen. Separates `kl_loss_unit_tests_passed` (Section 3, synthetic) from `kl_loss_integration_tested` (Section 4, real model). Manual-edit fallback instructions clarified: the dict is printed in this same cell, edit the literal values in the cell above and re-run.

In [16]:
DECISION_FIELDS = {
    "dedup_near_duplicates_found": None,
    "dedup_cross_dataset_pairs": None,
    "gaze_entropy_decision": None,
    "gaze_entropy_cliffs_delta": None,
    "kl_loss_unit_tests_passed": None,
    "kl_loss_integration_tested": None,
    "chosen_resolution": None,
    "chosen_batch_size": None,
    "chosen_grad_accum": None,
    "vram_headroom_factor": 0.7,
    "lora_trainable_pct": None,
    "literature_novelty_status": None,
    "random_seed": 42,
    "notes": "",
}

import json as _json

def _load_json(path):
    try:
        with open(path) as f:
            return _json.load(f)
    except FileNotFoundError:
        return None

decision_log = dict(DECISION_FIELDS)

entropy_result = _load_json("week1_entropy_decision.json")
if entropy_result:
    decision_log["gaze_entropy_decision"] = entropy_result["decision"]
    decision_log["gaze_entropy_cliffs_delta"] = entropy_result.get("cliffs_delta")

try:
    dupes_df = pd.read_csv("dedup_report.csv")
    decision_log["dedup_near_duplicates_found"] = len(dupes_df)
    decision_log["dedup_cross_dataset_pairs"] = int((dupes_df["dataset_a"] != dupes_df["dataset_b"]).sum())
except FileNotFoundError:
    decision_log["dedup_near_duplicates_found"] = 0

lora_result = _load_json("week1_lora_pct.json")
if lora_result:
    decision_log["lora_trainable_pct"] = lora_result["trainable_pct"]

try:
    vram_df = pd.read_csv("week1_vram_probe.csv")
    headroom = decision_log["vram_headroom_factor"]
    if "gpu_total_gb" in vram_df.columns and len(vram_df):
        gpu_total = vram_df["gpu_total_gb"].iloc[0]
        ok_rows = vram_df[(vram_df.status == "OK") & (vram_df.peak_gb < headroom * gpu_total)]
        if len(ok_rows):
            best = ok_rows.sort_values("peak_gb", ascending=False).iloc[0]
            decision_log["chosen_resolution"] = int(best.resolution)
            decision_log["chosen_batch_size"] = int(best.batch_size)
        else:
            print(f"WARNING: no (resolution,batch) combo fits under {headroom*100:.0f}% of {gpu_total:.1f} GB VRAM. "
                  "Consider lower resolution/batch or gradient checkpointing.")
except FileNotFoundError:
    pass

print("Auto-filled from Sections 1-7 where available:")
print(_json.dumps(decision_log, indent=2))

# --- Remaining manual fields ---
# EDIT THESE VALUES DIRECTLY, then re-run this cell:
decision_log["kl_loss_unit_tests_passed"] = True    # from Section 3
decision_log["kl_loss_integration_tested"] = True   # from Section 4

# Gradient accumulation does NOT change peak memory (Section 5's probe is unaffected by it) -
# it only changes effective batch size = chosen_batch_size * chosen_grad_accum. 8 is a
# starting point to reach effective batch ~16 from a memory-constrained micro-batch of 2;
# adjust based on what effective batch size you actually want for training stability.
decision_log["chosen_grad_accum"] = 8

decision_log["literature_novelty_status"] = "clear"  # "clear" | "needs reframing" | "investigate"
decision_log["notes"] = ""

with open("week1_decision_log.json", "w") as f:
    _json.dump(decision_log, f, indent=2)
print("\nSaved -> week1_decision_log.json")
print(_json.dumps(decision_log, indent=2))

try:
    import ipywidgets  # noqa
    print("\n(ipywidgets available, but a plain editable dict above is simpler and "
          "avoids the v2 confusion about which cell to edit - kept as-is.)")
except ImportError:
    pass

Auto-filled from Sections 1-7 where available:
{
  "dedup_near_duplicates_found": 237,
  "dedup_cross_dataset_pairs": 0,
  "gaze_entropy_decision": "DUAL_ARM",
  "gaze_entropy_cliffs_delta": 0.5594919786096256,
  "kl_loss_unit_tests_passed": null,
  "kl_loss_integration_tested": null,
  "chosen_resolution": 224,
  "chosen_batch_size": 32,
  "chosen_grad_accum": null,
  "vram_headroom_factor": 0.7,
  "lora_trainable_pct": 0.5108770699881195,
  "literature_novelty_status": null,
  "random_seed": 42,
  "notes": ""
}

Saved -> week1_decision_log.json
{
  "dedup_near_duplicates_found": 237,
  "dedup_cross_dataset_pairs": 0,
  "gaze_entropy_decision": "DUAL_ARM",
  "gaze_entropy_cliffs_delta": 0.5594919786096256,
  "kl_loss_unit_tests_passed": true,
  "kl_loss_integration_tested": true,
  "chosen_resolution": 224,
  "chosen_batch_size": 32,
  "chosen_grad_accum": 8,
  "vram_headroom_factor": 0.7,
  "lora_trainable_pct": 0.5108770699881195,
  "literature_novelty_status": "clear",
  "random_se